In [1]:
import pandas as pd 
import chromadb

In [2]:
from pathlib import Path

data_path = Path("data/university_faqs.csv")
df = pd.read_csv(data_path)

In [3]:
def load_faqs():
    df = pd.read_csv(DATA_PATH)

    print("Dataset loaded successfully!")
    print("Shape:", df.shape)
    print(df.head())

    return df

In [4]:
def clean_faqs(df):
    cleaned = df.copy()

    # Remove FAQs with missing questions or answers
    cleaned = cleaned.dropna(subset=["question", "answer"])

    # Remove extra whitespace
    for column in ["question", "answer", "category"]:
        cleaned[column] = cleaned[column].fillna("Unknown")
        cleaned[column] = cleaned[column].astype(str).str.strip()
        cleaned[column] = cleaned[column].str.replace(
            r"\s+", " ", regex=True
        )

    # Remove empty questions and answers
    cleaned = cleaned[
        (cleaned["question"] != "") &
        (cleaned["answer"] != "")
    ]

    # Remove repeated questions, ignoring case
    cleaned = cleaned.drop_duplicates(
        subset=["question"], keep="first"
    )

    cleaned = cleaned.reset_index(drop=True)

    return cleaned

In [5]:
def chunk_faqs(df, max_words=100):
    chunks = []

    for _, row in df.iterrows():
        text = f"Question: {row['question']} Answer: {row['answer']}"
        words = text.split()

        for start in range(0, len(words), max_words):
            chunk = " ".join(words[start:start + max_words])

            chunks.append({
                "faq_id": str(row["id"]),
                "category": row["category"],
                "text": chunk
            })

    return chunks

In [8]:
def store_embeddings(chunks):
    client = chromadb.PersistentClient(path="week-02/chroma_db")

    collection = client.get_or_create_collection(
        name="university_faqs"
    )

    collection.upsert(
        ids=[f"chunk_{i}" for i in range(len(chunks))],
        documents=[chunk["text"] for chunk in chunks],
        metadatas=[
            {
                "faq_id": chunk["faq_id"],
                "category": chunk["category"]
            }
            for chunk in chunks
        ]
    )

    print("Embeddings stored successfully!")
    print("Total stored chunks:", collection.count())

    return collection
    
   



In [9]:

def search_faqs(collection, query, top_k=3):
    results = collection.query(
        query_texts=[query],
        n_results=top_k
    )

    print(f"\nSearch query: {query}")
    print(f"\nTop {top_k} matches:\n")

    for i, (document, distance, metadata) in enumerate(
        zip(
            results["documents"][0],
            results["distances"][0],
            results["metadatas"][0]
        ),
        start=1
    ):
        print(f"Match {i}")
        print(f"Category: {metadata['category']}")
        print(f"FAQ ID: {metadata['faq_id']}")
        print(f"Text: {document}")
        print(f"Distance: {distance:.4f}")
        print("-" * 50)

In [14]:
# Load dataset
def load_faqs():
    df = pd.read_csv(DATA_PATH)


# Clean dataset
cleaned_df = clean_faqs(df)
print("\nCleaned dataset shape:", cleaned_df.shape)

# Chunk text
chunks = chunk_faqs(cleaned_df)
print("Total chunks:", len(chunks))

# Generate embeddings and store in ChromaDB
collection = store_embeddings(chunks)

# Search for similar FAQs
search_faqs(
    collection,
    "I forgot my university account password",
    top_k=3
)


Cleaned dataset shape: (15, 4)
Total chunks: 15
Embeddings stored successfully!
Total stored chunks: 15

Search query: I forgot my university account password

Top 3 matches:

Match 1
Category: Account
FAQ ID: 1
Text: Question: How do I reset my student portal password? Answer: Use the forgot password option on the student portal login page and follow the instructions sent to your registered email.
Distance: 0.6001
--------------------------------------------------
Match 2
Category: Technology
FAQ ID: 10
Text: Question: Where can I get help with my university email? Answer: Contact the campus IT help desk for assistance with your university email account.
Distance: 0.9266
--------------------------------------------------
Match 3
Category: Technology
FAQ ID: 14
Text: Question: How can I access online learning materials? Answer: Log in to the university learning management system using your assigned credentials.
Distance: 1.0522
--------------------------------------------------


In [15]:
search_faqs(
    collection,
    "How can I check my exam grades?",
    top_k=3
)


Search query: How can I check my exam grades?

Top 3 matches:

Match 1
Category: Academics
FAQ ID: 3
Text: Question: Where can I see my exam results? Answer: You can view your published examination results through the grades section of the student portal.
Distance: 0.6496
--------------------------------------------------
Match 2
Category: Academics
FAQ ID: 13
Text: Question: What should I do if my grades are incorrect? Answer: Contact your course instructor or department to request a grade review.
Distance: 0.9794
--------------------------------------------------
Match 3
Category: Academics
FAQ ID: 4
Text: Question: How do I get my academic transcript? Answer: Contact the university registrar office to request an official academic transcript.
Distance: 1.1000
--------------------------------------------------
